## Image Generating Pipeline for Stable Diffusion

https://huggingface.co/docs/diffusers/stable_diffusion

In [1]:
import torch
import math
from PIL import Image
from diffusers import StableDiffusionPipeline
from diffusers import DPMSolverMultistepScheduler
from diffusers import AutoencoderKL

In [ ]:
# StabDiff version
model_id = "runwayml/stable-diffusion-v1-5"

# Where to save the inferred images
#CONCEPTS_PATH = '../data/Wildbienen/Concepts_Bombus_vs_Andrena_StabDiff_color_texture/texture/smooth/'
#CONCEPTS_PATH = '../data/DanishFungi/Concepts_3spec/reddish_orange/'
CONCEPTS_PATH = '../data/Unsplash/prototypes/cheetah_proto/'
#CONCEPTS_PATH = '../data/Unsplash/concepts_stab/chequered/'
generator_seed = 1

### Some prompt engineering ###
### Wild Bees ###
#prompt = "Smooth and shiny dark brown bee, 4k, highly detailed"
#prompt = "Fluffy bee with very fuzzy yellow and black stripes, 4k, highly detailed"
#prompt = "Fluffy and very fuzzy dark orange bee, 4k, higly detailed"
#prompt = "Completely dark brown bee, 4k, highly detailed"
#prompt = "Completely dark orange bee, 4k, highly detailed"
#prompt = "Completely yellow bee, 4k, highly detailed"
#prompt = "Very fuzzy and fluffy bee, 4k, highly detailed"
#prompt = "Smooth shiny bee, 4k, highly detailed"

### Danish Fungi ###
#prompt = "Mushroom with white stem and bright red cap, 4k, highly detailed"
#prompt = "Mushroom with white stem and reddish orange to pinkish orange cap, 4k, highly detailed"
#prompt = "Mushroom with white stem and rose red to pink cap, 4k, highly detailed"

### ImageNet ###
#prompt = "White animal with black stripes, 4k, highly detailed"
prompt = "Animal with yellow fur and black dots, 4k, highly detailed"
#prompt = "Elongated chequered reptile, 4k, highly detailed"
#prompt = "Reptile with chequered shell, 4k, highly detailed"
#prompt = "Animal with white orange fur and black stripes, 4k, highly detailed"

### Broden ###
#prompt = "Chequered, 4k, highly detailed"

# Speedup 1: float16 precision instead of float32
# Speedup 2: inference steps
# Different schedulers/samplers require different numbers of inference steps for good results.
# The default sampler is `PNDMScheduler`, which needs around 50 steps.
# Other schedulers such as `DPMSolverMultistepScheduler` or `DPMSolverSinglestepScheduler`
# seem to get away with just 20 to 25 inference steps.
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda") # make inference on GPU
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config) # change scheduler

In [ ]:
# More memory optimization -> we can set batch size higher
pipe.enable_attention_slicing()

In [ ]:
# Quality improvement: replace default autoencoder -> images a bit sharper
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=torch.float16).to("cuda")
pipe.vae = vae

In [ ]:
# For generating different images with different seeds
offset = 0 # add b_size to this to generate new batches of images
# Batch inference
def get_inputs(batch_size=1):
    generator = [torch.Generator("cuda").manual_seed(generator_seed*(i+offset)) for i in range(batch_size)]
    prompts = batch_size * [prompt]
    num_inference_steps = 20
    return {"prompt": prompts, "generator": generator, "num_inference_steps": num_inference_steps}

# For plotting
def image_grid(imgs, rows=2, cols=2, save_imgs=False):
    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
        if save_imgs:
            img.save(CONCEPTS_PATH + str(i+offset) + '.jpg')
    return grid

In [ ]:
# Compute and display images
b_size, ncol = 30, 4
images = pipe(**get_inputs(batch_size=b_size)).images
image_grid(images, rows=int(math.ceil(b_size/ncol)), cols=ncol, save_imgs=True)